In [42]:
# !pip install pandas
# !pip install numpy
# !pip install matplotlib
# !pip install geopandas
# !pip install geoalchemy2
# !pip install sqlalchemy

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon, MultiPolygon
from geoalchemy2 import Geometry, WKTElement
import matplotlib.pyplot as plt
import os
import glob

import requests
from bs4 import BeautifulSoup
import html5lib

from sqlalchemy import create_engine
import psycopg2
import psycopg2.extras
import json

In [43]:
# !pip install psycopg2
# !pip install psycopg2-binary --user
# !pip install psycopg[binary]

credentials = "/Users/apple/Desktop/Data2901/Credentials.json"

def pgconnect(credential_filepath, db_schema="public"):
    with open(credential_filepath) as f:
        db_conn_dict = json.load(f)
        host       = db_conn_dict['host']
        db_user    = db_conn_dict['user']
        db_pw      = db_conn_dict['password']
        default_db = db_conn_dict['user']
        port       = db_conn_dict['port']
        try:
            db = create_engine(f'postgresql+psycopg2://{db_user}:{db_pw}@{host}:{port}/{default_db}', echo=False)
            conn = db.connect()
            print('Connected successfully.')
        except Exception as e:
            print("Unable to connect to the database.")
            print(e)
            db, conn = None, None
        return db,conn

def query(conn, sqlcmd, args=None, df=True):
    result = pd.DataFrame() if df else None
    try:
        if df:
            result = pd.read_sql_query(sqlcmd, conn, params=args)
        else:
            result = conn.execute(text(sqlcmd), args).fetchall()
            result = result[0] if len(result) == 1 else result
    except Exception as e:
        print("Error encountered: ", e, sep='\n')
    return result

In [44]:
db, conn = pgconnect(credentials)

Connected successfully.


In [45]:
query(conn, "select PostGIS_Version()")

,postgis_version
0,3.4 USE_GEOS=1 USE_PROJ=1 USE_STATS=1


# 1. import all the data

In [59]:
# get all the files in csv
path = os.getcwd() 
csv_files = glob.glob(os.path.join(path, "Data/*.csv")) 

print(csv_files)

# create empty list
dataframes_list = []
 
# append datasets into the list
for i in csv_files:
    temp_df = pd.read_csv(i)
    dataframes_list.append(temp_df)

df_income = dataframes_list[0]
df_pop = dataframes_list[1]
df_pol = dataframes_list[2]
df_business = dataframes_list[3]
print("Income: ",df_income.shape,'; ',df_income.columns)
print('Population: ',df_pop.shape,'; ',df_pop.columns)
print('Polling: ',df_pol.shape,'; ',df_pol.columns)
print('Business: ',df_business.shape,'; ',df_business.columns)

['/Users/apple/Desktop/Data2901/Assignment/DATA2901--2024S1/Data/Income.csv', '/Users/apple/Desktop/Data2901/Assignment/DATA2901--2024S1/Data/Population.csv', '/Users/apple/Desktop/Data2901/Assignment/DATA2901--2024S1/Data/PollingPlaces2019.csv', '/Users/apple/Desktop/Data2901/Assignment/DATA2901--2024S1/Data/Businesses.csv']
Income:  (642, 6) ;  Index(['sa2_code21', 'sa2_name', 'earners', 'median_age', 'median_income',
       'mean_income'],
      dtype='object')
Population:  (373, 21) ;  Index(['sa2_code', 'sa2_name', '0-4_people', '5-9_people', '10-14_people',
       '15-19_people', '20-24_people', '25-29_people', '30-34_people',
       '35-39_people', '40-44_people', '45-49_people', '50-54_people',
       '55-59_people', '60-64_people', '65-69_people', '70-74_people',
       '75-79_people', '80-84_people', '85-and-over_people', 'total_people'],
      dtype='object')
Polling:  (2930, 17) ;  Index(['FID', 'state', 'division_id', 'division_name', 'polling_place_id',
       'polling_pl

In [60]:
df_income.head()

,sa2_code21,sa2_name,earners,median_age,median_income,mean_income
0,101021007,Braidwood,2467,51,46640,68904
1,101021008,Karabar,5103,42,65564,69672
2,101021009,Queanbeyan,7028,39,63528,69174
3,101021010,Queanbeyan - East,3398,39,66148,74162
4,101021012,Queanbeyan West - Jerrabomberra,8422,44,78630,91981


In [61]:
df_pop.head()

,sa2_code,sa2_name,0-4_people,5-9_people,10-14_people,15-19_people,20-24_people,25-29_people,30-34_people,35-39_people,...,45-49_people,50-54_people,55-59_people,60-64_people,65-69_people,70-74_people,75-79_people,80-84_people,85-and-over_people,total_people
0,102011028,Avoca Beach - Copacabana,424,522,623,552,386,222,306,416,...,572,602,570,520,464,369,226,142,70,7530
1,102011029,Box Head - MacMasters Beach,511,666,702,592,461,347,420,535,...,749,749,794,895,863,925,603,331,264,11052
2,102011030,Calga - Kulnura,200,225,258,278,274,227,214,286,...,325,436,422,397,327,264,190,100,75,4748
3,102011031,Erina - Green Point,683,804,880,838,661,502,587,757,...,859,882,901,930,917,1065,976,773,1028,14803
4,102011032,Gosford - Springfield,1164,1044,1084,1072,1499,1864,1750,1520,...,1330,1241,1377,1285,1166,949,664,476,537,21346


In [64]:
df_pol.tail()
df_pol['the_geom']

0                                   NaN
1                                   NaN
2                                   NaN
3                                   NaN
4                                   NaN
                     ...               
2925           POINT (-34.5642 150.858)
2926    POINT (-34.5508228 150.8597546)
2927           POINT (-34.4409 150.424)
2928           POINT (-34.5316 150.866)
2929                                NaN
Name: the_geom, Length: 2930, dtype: object

In [63]:
df_business.head()

,industry_code,industry_name,sa2_code,sa2_name,0_to_50k_businesses,50k_to_200k_businesses,200k_to_2m_businesses,2m_to_5m_businesses,5m_to_10m_businesses,10m_or_more_businesses,total_businesses
0,A,"Agriculture, Forestry and Fishing",101021007,Braidwood,136,92,63,4,0,0,296
1,A,"Agriculture, Forestry and Fishing",101021008,Karabar,6,3,0,0,0,0,9
2,A,"Agriculture, Forestry and Fishing",101021009,Queanbeyan,6,4,3,0,0,3,15
3,A,"Agriculture, Forestry and Fishing",101021010,Queanbeyan - East,0,3,0,0,0,0,3
4,A,"Agriculture, Forestry and Fishing",101021012,Queanbeyan West - Jerrabomberra,7,4,5,0,0,0,16


## import txt files

In [51]:
# import txt file
# path = os.getcwd() 
txt_files = glob.glob(os.path.join(path, "*/*.txt")) 
print(txt_files[0])
i = txt_files[0]
df_stops = pd.read_csv(i)
print(df_stops.shape)
print(df_stops.columns)
df_stops.tail()

/Users/apple/Desktop/Data2901/Assignment/DATA2901--2024S1/Data/Stops.txt
(114718, 9)
Index(['stop_id', 'stop_code', 'stop_name', 'stop_lat', 'stop_lon',
       'location_type', 'parent_station', 'wheelchair_boarding',
       'platform_code'],
      dtype='object')


,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,platform_code
114713,212753,212753.0,"Sydney Olympic Park Wharf, Side B",-33.822016,151.078797,NaN,21271,1,B
114714,2137185,2137185.0,"Cabarita Wharf, Side A",-33.840669,151.116926,NaN,21371,1,1A
114715,2137186,2137186.0,"Cabarita Wharf, Side B",-33.840769,151.116899,NaN,21371,1,1B
114716,21501,21501.0,Parramatta Wharf,-33.813904,151.010577,NaN,2150112,1,NaN
114717,2150112,NaN,Parramatta Wharf,-33.813952,151.010482,1.0,NaN,1,NaN


## read the shp file from catchments

In [89]:
# get the shp files
print(path)
shp_files = glob.glob(os.path.join(path, "Data/*/*.shp")) 

# create empty list
dataframes_list_shp = []
 
# append datasets into the list
for i in shp_files:
    print(i)
    temp_df = gpd.read_file(i)
    dataframes_list_shp.append(temp_df)

df_catch_f = dataframes_list_shp[1]
df_catch_s = dataframes_list_shp[0]
df_catch_p = dataframes_list_shp[2]
df_sa2 = dataframes_list_shp[3]
print("catchment future: ",df_catch_f.shape,'; ',df_catch_f.columns)
print('catchment secondary: ',df_catch_s.shape,'; ',df_catch_s.columns)
print('catchment primary: ',df_catch_p.shape,'; ',df_catch_p.columns)
print('sa2: ',df_sa2.shape,'; ',df_sa2.columns)

df_catch_f.head()

/Users/apple/Desktop/Data2901/Assignment/DATA2901--2024S1
/Users/apple/Desktop/Data2901/Assignment/DATA2901--2024S1/Data/catchments/catchments_secondary.shp
/Users/apple/Desktop/Data2901/Assignment/DATA2901--2024S1/Data/catchments/catchments_future.shp
/Users/apple/Desktop/Data2901/Assignment/DATA2901--2024S1/Data/catchments/catchments_primary.shp
/Users/apple/Desktop/Data2901/Assignment/DATA2901--2024S1/Data/SA2_2021_AUST_SHP_GDA2020/SA2_2021_AUST_GDA2020.shp
catchment future:  (436, 19) ;  Index(['USE_ID', 'CATCH_TYPE', 'USE_DESC', 'ADD_DATE', 'KINDERGART', 'YEAR1',
       'YEAR2', 'YEAR3', 'YEAR4', 'YEAR5', 'YEAR6', 'YEAR7', 'YEAR8', 'YEAR9',
       'YEAR10', 'YEAR11', 'YEAR12', 'PRIORITY', 'geometry'],
      dtype='object')
catchment secondary:  (30, 18) ;  Index(['USE_ID', 'CATCH_TYPE', 'USE_DESC', 'ADD_DATE', 'KINDERGART', 'YEAR1',
       'YEAR2', 'YEAR3', 'YEAR4', 'YEAR5', 'YEAR6', 'YEAR7', 'YEAR8', 'YEAR9',
       'YEAR10', 'YEAR11', 'YEAR12', 'geometry'],
      dtype='object')

,USE_ID,CATCH_TYPE,USE_DESC,ADD_DATE,KINDERGART,YEAR1,YEAR2,YEAR3,YEAR4,YEAR5,YEAR6,YEAR7,YEAR8,YEAR9,YEAR10,YEAR11,YEAR12,PRIORITY,geometry
0,8503,HIGH_COED,Billabong HS,20200507,N,N,N,N,N,N,N,Y,Y,Y,Y,Y,Y,None,"POLYGON ((146.67182 -35.31444, 146.68930 -35.3..."
1,8266,HIGH_COED,James Fallon HS,20200507,N,N,N,N,N,N,N,Y,Y,Y,Y,Y,Y,None,"POLYGON ((147.08734 -35.86271, 147.10413 -35.8..."
2,8505,HIGH_COED,Murray HS,20200507,N,N,N,N,N,N,N,Y,Y,Y,Y,Y,Y,None,"POLYGON ((146.81448 -35.78341, 146.81250 -35.7..."
3,8458,HIGH_COED,Kingswood HS,20201016,N,N,N,N,N,N,N,Y,Y,Y,Y,Y,Y,None,"MULTIPOLYGON (((150.68600 -33.74031, 150.68631..."
4,8559,HIGH_COED,Jamison HS,20201016,N,N,N,N,N,N,N,Y,Y,Y,Y,Y,Y,None,"POLYGON ((150.69513 -33.75627, 150.68936 -33.7..."


In [90]:
df_sa2.head()

,SA2_CODE21,SA2_NAME21,CHG_FLAG21,CHG_LBL21,SA3_CODE21,SA3_NAME21,SA4_CODE21,SA4_NAME21,GCC_CODE21,GCC_NAME21,STE_CODE21,STE_NAME21,AUS_CODE21,AUS_NAME21,AREASQKM21,LOCI_URI21,geometry
0,101021007,Braidwood,0,No change,10102,Queanbeyan,101,Capital Region,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,3418.3525,http://linked.data.gov.au/dataset/asgsed3/SA2/...,"POLYGON ((149.58424 -35.44426, 149.58444 -35.4..."
1,101021008,Karabar,0,No change,10102,Queanbeyan,101,Capital Region,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,6.9825,http://linked.data.gov.au/dataset/asgsed3/SA2/...,"POLYGON ((149.21899 -35.36738, 149.21800 -35.3..."
2,101021009,Queanbeyan,0,No change,10102,Queanbeyan,101,Capital Region,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,4.7620,http://linked.data.gov.au/dataset/asgsed3/SA2/...,"POLYGON ((149.21326 -35.34325, 149.21619 -35.3..."
3,101021010,Queanbeyan - East,0,No change,10102,Queanbeyan,101,Capital Region,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,13.0032,http://linked.data.gov.au/dataset/asgsed3/SA2/...,"POLYGON ((149.24034 -35.34781, 149.24024 -35.3..."
4,101021012,Queanbeyan West - Jerrabomberra,0,No change,10102,Queanbeyan,101,Capital Region,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,13.6748,http://linked.data.gov.au/dataset/asgsed3/SA2/...,"POLYGON ((149.19572 -35.36126, 149.19970 -35.3..."


In [91]:
df_sa2.dtypes

SA2_CODE21      object
SA2_NAME21      object
CHG_FLAG21      object
CHG_LBL21       object
SA3_CODE21      object
SA3_NAME21      object
SA4_CODE21      object
SA4_NAME21      object
GCC_CODE21      object
GCC_NAME21      object
STE_CODE21      object
STE_NAME21      object
AUS_CODE21      object
AUS_NAME21      object
AREASQKM21     float64
LOCI_URI21      object
geometry      geometry
dtype: object

In [54]:
df_catch_s['PRIORITY'] = 'None'
df_catch_s1 = df_catch_s[['USE_ID', 'CATCH_TYPE', 'USE_DESC', 'ADD_DATE', 'KINDERGART', 'YEAR1',
       'YEAR2', 'YEAR3', 'YEAR4', 'YEAR5', 'YEAR6', 'YEAR7', 'YEAR8', 'YEAR9',
       'YEAR10', 'YEAR11', 'YEAR12', 'PRIORITY', 'geometry']]

In [55]:
df_school = pd.concat([df_catch_f, df_catch_s1,df_catch_p])
print(df_school.shape)
df_school.tail()

(2128, 19)


,USE_ID,CATCH_TYPE,USE_DESC,ADD_DATE,KINDERGART,YEAR1,YEAR2,YEAR3,YEAR4,YEAR5,YEAR6,YEAR7,YEAR8,YEAR9,YEAR10,YEAR11,YEAR12,PRIORITY,geometry
1657,4383,PRIMARY,E A Southee PS,20200315,Y,Y,Y,Y,Y,Y,Y,N,N,N,N,N,N,None,"POLYGON ((147.94621 -34.55863, 147.95292 -34.5..."
1658,3275,PRIMARY,Tumbarumba PS,20200507,Y,Y,Y,Y,Y,Y,Y,N,N,N,N,N,N,None,"POLYGON ((148.12885 -35.60082, 148.23155 -35.6..."
1659,2239,PRIMARY,Jindera PS,20200507,Y,Y,Y,Y,Y,Y,Y,N,N,N,N,N,N,None,"POLYGON ((146.86148 -35.87511, 146.87402 -35.8..."
1660,3594,PRIMARY,Louth PS,20200604,Y,Y,Y,Y,Y,Y,Y,N,N,N,N,N,N,None,"POLYGON ((145.18403 -29.65805, 145.18434 -29.6..."
1661,2155,PRIMARY,Hermidale PS,20200604,Y,Y,Y,Y,Y,Y,Y,N,N,N,N,N,N,None,"POLYGON ((146.46001 -31.31688, 146.47453 -31.3..."


In [56]:
#transform to multipolygon
srid = 4326

def create_wkt_element(geom, srid):
    if geom.geom_type == 'Polygon':
        geom = MultiPolygon([geom])
    return WKTElement(geom.wkt, srid)

df_schoolog = df_school.copy()  # creating a copy of the original for laterog = df_school.copy()  # creating a copy of the original for later
df_school['geom'] = df_school['geometry'].apply(lambda x: create_wkt_element(geom=x,srid=srid))  # applying the function
df_school = df_school.drop(columns="geometry")  # deleting the old copy
df_school

,USE_ID,CATCH_TYPE,USE_DESC,ADD_DATE,KINDERGART,YEAR1,YEAR2,YEAR3,YEAR4,YEAR5,YEAR6,YEAR7,YEAR8,YEAR9,YEAR10,YEAR11,YEAR12,PRIORITY,geom
0,8503,HIGH_COED,Billabong HS,20200507,N,N,N,N,N,N,N,Y,Y,Y,Y,Y,Y,None,MULTIPOLYGON (((146.67182402032344 -35.3144375...
1,8266,HIGH_COED,James Fallon HS,20200507,N,N,N,N,N,N,N,Y,Y,Y,Y,Y,Y,None,MULTIPOLYGON (((147.08733806259178 -35.8627146...
2,8505,HIGH_COED,Murray HS,20200507,N,N,N,N,N,N,N,Y,Y,Y,Y,Y,Y,None,MULTIPOLYGON (((146.81447829547324 -35.7834062...
3,8458,HIGH_COED,Kingswood HS,20201016,N,N,N,N,N,N,N,Y,Y,Y,Y,Y,Y,None,MULTIPOLYGON (((150.68599834118749 -33.7403060...
4,8559,HIGH_COED,Jamison HS,20201016,N,N,N,N,N,N,N,Y,Y,Y,Y,Y,Y,None,MULTIPOLYGON (((150.69513440644116 -33.7562688...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1657,4383,PRIMARY,E A Southee PS,20200315,Y,Y,Y,Y,Y,Y,Y,N,N,N,N,N,N,None,MULTIPOLYGON (((147.9462089946497 -34.55863148...
1658,3275,PRIMARY,Tumbarumba PS,20200507,Y,Y,Y,Y,Y,Y,Y,N,N,N,N,N,N,None,MULTIPOLYGON (((148.12885348977485 -35.6008184...
1659,2239,PRIMARY,Jindera PS,20200507,Y,Y,Y,Y,Y,Y,Y,N,N,N,N,N,N,None,MULTIPOLYGON (((146.86147943204122 -35.8751106...
1660,3594,PRIMARY,Louth PS,20200604,Y,Y,Y,Y,Y,Y,Y,N,N,N,N,N,N,None,MULTIPOLYGON (((145.18402754685187 -29.6580498...


In [ ]:
# df_sa2['geometry'].geom_type == 'Polygon'
df_sa2.dtypes

SA2_CODE21      object
SA2_NAME21      object
CHG_FLAG21      object
CHG_LBL21       object
SA3_CODE21      object
SA3_NAME21      object
SA4_CODE21      object
SA4_NAME21      object
GCC_CODE21      object
GCC_NAME21      object
STE_CODE21      object
STE_NAME21      object
AUS_CODE21      object
AUS_NAME21      object
AREASQKM21     float64
LOCI_URI21      object
geometry      geometry
dtype: object

In [ ]:
df_sa2og = df_sa2.copy()  # creating a copy of the original for laterog = df_sa2.copy()  # creating a copy of the original for later
df_sa2 = df_sa2[df_sa2.geometry.type == 'Polygon']
df_sa2['geom'] = df_sa2['geometry'].apply(lambda x: create_wkt_element(geom=x,srid=srid))  # applying the function
df_sa2 = df_sa2.drop(columns="geometry")  # deleting the old copy
df_sa2['SA2_CODE21'] = df_sa2['SA2_CODE21'].astype(int)
df_sa2 

/Applications/anaconda3/lib/python3.8/site-packages/geopandas/geodataframe.py:1538: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,SA2_CODE21,SA2_NAME21,CHG_FLAG21,CHG_LBL21,SA3_CODE21,SA3_NAME21,SA4_CODE21,SA4_NAME21,GCC_CODE21,GCC_NAME21,STE_CODE21,STE_NAME21,AUS_CODE21,AUS_NAME21,AREASQKM21,LOCI_URI21,geom
0,101021007,Braidwood,0,No change,10102,Queanbeyan,101,Capital Region,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,3418.3525,http://linked.data.gov.au/dataset/asgsed3/SA2/...,MULTIPOLYGON (((149.58423846300806 -35.4442571...
1,101021008,Karabar,0,No change,10102,Queanbeyan,101,Capital Region,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,6.9825,http://linked.data.gov.au/dataset/asgsed3/SA2/...,MULTIPOLYGON (((149.2189874391411 -35.36738117...
2,101021009,Queanbeyan,0,No change,10102,Queanbeyan,101,Capital Region,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,4.7620,http://linked.data.gov.au/dataset/asgsed3/SA2/...,MULTIPOLYGON (((149.21326493309647 -35.3432452...
3,101021010,Queanbeyan - East,0,No change,10102,Queanbeyan,101,Capital Region,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,13.0032,http://linked.data.gov.au/dataset/asgsed3/SA2/...,MULTIPOLYGON (((149.2403376383506 -35.34780977...
4,101021012,Queanbeyan West - Jerrabomberra,0,No change,10102,Queanbeyan,101,Capital Region,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,13.6748,http://linked.data.gov.au/dataset/asgsed3/SA2/...,MULTIPOLYGON (((149.19572324350193 -35.3612624...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2460,801101145,Molonglo - East,1,New,80110,Molonglo,801,Australian Capital Territory,8ACTE,Australian Capital Territory,8,Australian Capital Territory,AUS,Australia,7.3323,http://linked.data.gov.au/dataset/asgsed3/SA2/...,MULTIPOLYGON (((149.05296755302768 -35.3015961...
2461,801101146,Whitlam,1,New,80110,Molonglo,801,Australian Capital Territory,8ACTE,Australian Capital Territory,8,Australian Capital Territory,AUS,Australia,3.2590,http://linked.data.gov.au/dataset/asgsed3/SA2/...,MULTIPOLYGON (((149.05113657739636 -35.2692051...
2462,801111140,ACT - South West,0,No change,80111,Uriarra - Namadgi,801,Australian Capital Territory,8ACTE,Australian Capital Territory,8,Australian Capital Territory,AUS,Australia,416.7824,http://linked.data.gov.au/dataset/asgsed3/SA2/...,MULTIPOLYGON (((148.8838115837419 -35.26411249...
2463,801111141,Namadgi,0,No change,80111,Uriarra - Namadgi,801,Australian Capital Territory,8ACTE,Australian Capital Territory,8,Australian Capital Territory,AUS,Australia,1202.7527,http://linked.data.gov.au/dataset/asgsed3/SA2/...,MULTIPOLYGON (((148.8040699920124 -35.37619101...


# 2. data validations

In [ ]:
## check data types, nulls, duplicates, unique values
df_school.dtypes

USE_ID        object
CATCH_TYPE    object
USE_DESC      object
ADD_DATE      object
KINDERGART    object
YEAR1         object
YEAR2         object
YEAR3         object
YEAR4         object
YEAR5         object
YEAR6         object
YEAR7         object
YEAR8         object
YEAR9         object
YEAR10        object
YEAR11        object
YEAR12        object
PRIORITY      object
geom          object
dtype: object

In [ ]:
print(df_school.shape)
print(df_school.drop_duplicates().shape)

print(df_business.shape)
print(df_business.drop_duplicates().shape)

print(df_income.shape)
print(df_income.drop_duplicates().shape)

print(df_pol.shape)
print(df_pol.drop_duplicates().shape)

print(df_pop.shape)
print(df_pop.drop_duplicates().shape)
# no duplicated rows

(2128, 19)
(2128, 19)
(12217, 11)
(12217, 11)
(642, 6)
(642, 6)
(2930, 17)
(2930, 17)
(373, 21)
(373, 21)


In [ ]:
print(df_school.isna().sum())
df_school.nunique()

USE_ID           0
CATCH_TYPE       0
USE_DESC         0
ADD_DATE       391
KINDERGART       0
YEAR1            0
YEAR2            0
YEAR3            0
YEAR4            0
YEAR5            0
YEAR6            0
YEAR7            0
YEAR8            0
YEAR9            0
YEAR10           0
YEAR11           0
YEAR12           0
PRIORITY      2087
geom             0
dtype: int64


USE_ID        2035
CATCH_TYPE       7
USE_DESC      2035
ADD_DATE       209
KINDERGART       4
YEAR1            4
YEAR2            4
YEAR3            4
YEAR4            4
YEAR5            4
YEAR6            4
YEAR7            4
YEAR8            4
YEAR9            4
YEAR10           5
YEAR11           5
YEAR12           6
PRIORITY         3
geom          2084
dtype: int64

In [ ]:
print(df_business.dtypes)
print(df_business.isna().sum())
df_business.nunique()

industry_code             object
industry_name             object
sa2_code                   int64
sa2_name                  object
0_to_50k_businesses        int64
50k_to_200k_businesses     int64
200k_to_2m_businesses      int64
2m_to_5m_businesses        int64
5m_to_10m_businesses       int64
10m_or_more_businesses     int64
total_businesses           int64
dtype: object
industry_code             0
industry_name             0
sa2_code                  0
sa2_name                  0
0_to_50k_businesses       0
50k_to_200k_businesses    0
200k_to_2m_businesses     0
2m_to_5m_businesses       0
5m_to_10m_businesses      0
10m_or_more_businesses    0
total_businesses          0
dtype: int64


industry_code              19
industry_name              19
sa2_code                  643
sa2_name                  643
0_to_50k_businesses       249
50k_to_200k_businesses    260
200k_to_2m_businesses     269
2m_to_5m_businesses        79
5m_to_10m_businesses       44
10m_or_more_businesses     67
total_businesses          561
dtype: int64

In [ ]:
print(df_pol.dtypes)
print(df_pol.isna().sum())
df_pol.nunique()

FID                             object
state                           object
division_id                      int64
division_name                   object
polling_place_id                 int64
polling_place_type_id            int64
polling_place_name              object
premises_name                   object
premises_address_1              object
premises_address_2              object
premises_address_3              object
premises_suburb                 object
premises_state_abbreviation     object
premises_post_code             float64
latitude                       float64
longitude                      float64
the_geom                        object
dtype: object
FID                               0
state                             0
division_id                       0
division_name                     0
polling_place_id                  0
polling_place_type_id             0
polling_place_name                0
premises_name                     0
premises_address_1              193

FID                            2930
state                             1
division_id                      47
division_name                    47
polling_place_id               2930
polling_place_type_id             4
polling_place_name             2757
premises_name                  2463
premises_address_1             2408
premises_address_2               46
premises_address_3                7
premises_suburb                1522
premises_state_abbreviation       3
premises_post_code              593
latitude                       2498
longitude                      2242
the_geom                       2543
dtype: int64

In [ ]:
print(df_pop.isna().sum())
df_pop.nunique()

sa2_code              0
sa2_name              0
0-4_people            0
5-9_people            0
10-14_people          0
15-19_people          0
20-24_people          0
25-29_people          0
30-34_people          0
35-39_people          0
40-44_people          0
45-49_people          0
50-54_people          0
55-59_people          0
60-64_people          0
65-69_people          0
70-74_people          0
75-79_people          0
80-84_people          0
85-and-over_people    0
total_people          0
dtype: int64


sa2_code              373
sa2_name              373
0-4_people            324
5-9_people            327
10-14_people          332
15-19_people          318
20-24_people          324
25-29_people          340
30-34_people          336
35-39_people          331
40-44_people          328
45-49_people          326
50-54_people          326
55-59_people          317
60-64_people          314
65-69_people          304
70-74_people          302
75-79_people          286
80-84_people          258
85-and-over_people    272
total_people          365
dtype: int64

In [ ]:
print(df_business.dtypes)
print(df_business.isna().sum())
df_business.nunique()

industry_code             object
industry_name             object
sa2_code                   int64
sa2_name                  object
0_to_50k_businesses        int64
50k_to_200k_businesses     int64
200k_to_2m_businesses      int64
2m_to_5m_businesses        int64
5m_to_10m_businesses       int64
10m_or_more_businesses     int64
total_businesses           int64
dtype: object
industry_code             0
industry_name             0
sa2_code                  0
sa2_name                  0
0_to_50k_businesses       0
50k_to_200k_businesses    0
200k_to_2m_businesses     0
2m_to_5m_businesses       0
5m_to_10m_businesses      0
10m_or_more_businesses    0
total_businesses          0
dtype: int64


industry_code              19
industry_name              19
sa2_code                  643
sa2_name                  643
0_to_50k_businesses       249
50k_to_200k_businesses    260
200k_to_2m_businesses     269
2m_to_5m_businesses        79
5m_to_10m_businesses       44
10m_or_more_businesses     67
total_businesses          561
dtype: int64

In [ ]:
# df_income.dtypes
df_income.head()

,sa2_code21,sa2_name,earners,median_age,median_income,mean_income
0,101021007,Braidwood,2467,51,46640,68904
1,101021008,Karabar,5103,42,65564,69672
2,101021009,Queanbeyan,7028,39,63528,69174
3,101021010,Queanbeyan - East,3398,39,66148,74162
4,101021012,Queanbeyan West - Jerrabomberra,8422,44,78630,91981


In [ ]:
# Change column A's values to floats
# df_income['earners'].sort_values().unique
df_income[df_income.earners == 'np']

,sa2_code21,sa2_name,earners,median_age,median_income,mean_income
76,103031075,Wollangambe - Wollemi,np,np,np,np
135,107011133,Port Kembla Industrial,np,np,np,np
140,107021135,Illawarra Catchment Reserve,np,np,np,np
167,108031161,Lord Howe Island,np,np,np,np
373,118011342,Centennial Park,np,np,np,np
507,123021439,Holsworthy Military Area,np,np,np,np
526,124021456,Blue Mountains - South,np,np,np,np


In [ ]:
# keep columns only when valid data exists -- drop 7 columns due to no valid value in earners and median age/income
# Change column values to integers
df_income = df_income[df_income.earners != 'np'].astype({'earners': int, 'median_age': int,'median_income': int,'mean_income': int})


print(df_income.dtypes)

sa2_code21        int64
sa2_name         object
earners           int64
median_age        int64
median_income     int64
mean_income       int64
dtype: object


# 3. import into sql 

In [ ]:
conn.execute("""
DROP TABLE IF EXISTS polls;
CREATE TABLE polls (
    FID                             varchar(100),
state                           varchar(100),
division_id                      integer primary key,
division_name                   varchar(100),
polling_place_id                 integer,
polling_place_type_id            integer,
polling_place_name              varchar(100),
premises_name                   varchar(100),
premises_address_1              varchar(100),
premises_address_2              varchar(100),
premises_address_3              varchar(100),
premises_suburb                 varchar(100),
premises_state_abbreviation     varchar(100),
premises_post_code             integer,
latitude                       float,
longitude                      float,
the_geom                       GEOMETRY(POINT,4326)
);"""
)
df_pol.to_sql('polls', conn, if_exists='replace', index=False, dtype={'geom': Geometry('POINT', srid)})
query(conn, "select * from polls")

conn.execute("""
DROP TABLE IF EXISTS populations;
CREATE TABLE populations (
    pop_est NUMERIC, 
    continent VARCHAR(80), 
    name VARCHAR(80), 
    iso_a3 VARCHAR(80), 
    gdp_md_est NUMERIC,
    geom GEOMETRY(MULTIPOLYGON,4326) primary key
);"""
)
df_pop.to_sql('populations', conn, if_exists='replace', index=False, dtype={'geom': Geometry('POINT', srid)})
query(conn, "select * from populations")

conn.execute("""
DROP TABLE IF EXISTS incomes;
CREATE TABLE incomes (
    sa2_code21        integer primary key,
sa2_name         varchar(100),
earners          integer,
median_age       integer,
median_income    integer,
mean_income      integer
);"""
)
df_income.to_sql('incomes', conn, if_exists='replace', index=False, dtype={'geom': Geometry('POINT', srid)})
query(conn, "select * from incomes")

conn.execute("""
DROP TABLE IF EXISTS schools;
CREATE TABLE schools (
USE_ID        integer,
CATCH_TYPE    varchar(100),
USE_DESC      varchar(100),
ADD_DATE      date,
KINDERGART    varchar(100),
YEAR1         varchar(100),
YEAR2         varchar(100),
YEAR3         varchar(100),
YEAR4         varchar(100),
YEAR5         varchar(100),
YEAR6         varchar(100),
YEAR7         varchar(100),
YEAR8         varchar(100),
YEAR9         varchar(100),
YEAR10        varchar(100),
YEAR11        varchar(100),
YEAR12        varchar(100),
PRIORITY      varchar(100),
    geom GEOMETRY(MULTIPOLYGON,4326) primary key
);"""
)
df_school.to_sql('schools', conn, if_exists='replace', index=False, dtype={'geom': Geometry('MULTIPOLYGON', srid)})
query(conn, "select * from schools")


conn.execute("""
DROP TABLE IF EXISTS business;
CREATE TABLE business (
    industry_code             varchar(100),
    industry_name             varchar(100),
    sa2_code                   integer,
    sa2_name                  varchar(100),
    "0_to_50k_businesses"       integer,
    "50k_to_200k_businesses"     integer,
    "200k_to_2m_businesses"      integer,
    "2m_to_5m_businesses"        integer,
    "5m_to_10m_businesses"       integer,
    "10m_or_more_businesses"     integer,
    total_businesses           integer
);"""
)
df_business.to_sql('business', conn, if_exists='replace', index=False, dtype={'geom': Geometry('POINT', srid)})
query(conn, "select * from business")


conn.execute("""
DROP TABLE IF EXISTS stops;
CREATE TABLE stops (
stop_id int primary key, 
stop_code int, 
stop_name varchar(100), 
stop_lat float, 
stop_lon float,
location_type int, 
parent_station int, 
wheelchair_boarding int,
platform_code varchar(100)
);"""
)
df_stops.to_sql('stops', conn, if_exists='replace', index=False)
query(conn, "select * from stops")


conn.execute("""
DROP TABLE IF EXISTS sa2;
CREATE TABLE sa2 (
SA2_CODE21      integer primary key,
SA2_NAME21      varchar(100),
CHG_FLAG21      varchar(100),
CHG_LBL21       varchar(100),
SA3_CODE21      integer,
SA3_NAME21      varchar(100),
SA4_CODE21      integer,
SA4_NAME21      varchar(100),
GCC_CODE21      varchar(100),
GCC_NAME21      varchar(100),
STE_CODE21      integer,
STE_NAME21      varchar(100),
AUS_CODE21      varchar(100),
AUS_NAME21      varchar(100),
AREASQKM21     float,
LOCI_URI21      varchar(100),
geometry      geometry(Multipolygon,4326)
             );"""
)
df_sa2.to_sql('sa2', conn, if_exists='replace', index=False, dtype={'geom': Geometry('Multipolygon', srid)})
query(conn, "select * from sa2")

In [ ]:
df_business

## Task 2
Compute a score for how ”well-resourced” each individual neighbourhood is according to the formula provided on the next page, where S is the sigmoid function, z is the normalised z-score, and ’young people’ are defined as anyone aged 0-19.


### 2.1 Merge into 1 big dataset based on SA2Codes (except schools)

In [ ]:
conn.execute('''
drop table if exists combined;
    create table combined as
	with polls as (select *,ST_SetSRID(st_makepoint("longitude","latitude"),4326) as geom 
    			from polls
			   where the_geom is not null )
    ,stops as (select *,ST_SetSRID(st_makepoint("stop_lon","stop_lat"),4326) as geom 
    			from stops
			   where stop_lat is not null and stop_lon is not null)
	,join_polls as (select s."SA2_CODE21",s."SA2_NAME21",s."SA3_NAME21",s."SA4_NAME21","STE_NAME21",s."GCC_NAME21", s.geom,ST_Area(s.geom) * 1000000 as area_size,count(distinct p.*) as num_polls
				   from sa2 s
				   left join polls p on st_contains(s.geom, p.geom) -- 2505
				   
					group by 1,2,3,4,5,6,7,8
				   )
	,join_schools as (
		select s."SA2_CODE21",s."SA2_NAME21",s."SA3_NAME21",s."SA4_NAME21","STE_NAME21",s."GCC_NAME21", s.geom, s.area_size, num_polls,count(distinct sch."USE_ID") as num_schools
				   from join_polls s
				   left join schools sch on st_intersects(s.geom,sch.geom)
					  group by 1,2,3,4,5,6,7,8,9
				   )
	,join_stops as (select s."SA2_CODE21",s."SA2_NAME21",s."SA3_NAME21",s."SA4_NAME21","STE_NAME21",s."GCC_NAME21", s.geom, s.area_size, num_polls,num_schools,count(distinct stop_id) as num_stops
				   from join_schools s
				   left join stops on st_contains(s.geom, stops.geom)
					  group by 1,2,3,4,5,6,7,8,9,10
				   )
select 
s."SA2_CODE21",s."SA2_NAME21",s."SA3_NAME21",s."SA4_NAME21","STE_NAME21",s."GCC_NAME21", s.geom, s.area_size, num_polls,num_schools,num_stops
,sum(i.earners) as num_earners
,avg(i.median_age) as median_age
,avg(i.median_income) as median_income
,avg(i.mean_income) as mean_income
,sum(b.total_businesses) as total_businesses
,sum("0_to_50k_businesses" * 25 + "50k_to_200k_businesses" * 125 + "200k_to_2m_businesses" * 1100 + "2m_to_5m_businesses" * 3500 + "5m_to_10m_businesses" * 7500 + "10m_or_more_businesses" * 12000) as Total_turnover
,sum(ppl."0-4_people"+ppl."5-9_people" + ppl."10-14_people"+ppl."15-19_people") as num_0_19_people
,case when sum("total_people") =0 then null else sum(ppl."0-4_people"+ppl."5-9_people" + ppl."10-14_people"+ppl."15-19_people") / sum(ppl."total_people") end as Young_perc
,sum("total_people") total_people

from join_stops s
left join business b on b.sa2_code = s."SA2_CODE21" -- inner 11229
left join populations ppl on ppl.sa2_code = s."SA2_CODE21"-- inner 359
left join incomes i on i."sa2_code21" = s."SA2_CODE21" --inner 591
where num_polls is not null or num_schools is not null or num_stops is not null
or i.earners is not null or b.total_businesses is not null or ppl.total_people is not null
group by 1,2,3,4,5,6,7,8,9,10,11
''')
df_combined = query(conn, "select * from combined")

In [ ]:
df_combined.head()

In [ ]:
df_combined.dtypes

### check columns and understand the ranges

In [ ]:
df_combined.describe()

### check distributions

In [ ]:
df_combined['area_size'].hist(bins=100)

# add labels and title
plt.xlabel('area_size')
plt.ylabel('Frequency')
plt.title('Distribution of area_size')
# long tail

In [ ]:
df_combined['num_polls'].hist(bins=100)

# add labels and title
plt.xlabel('num_polls')
plt.ylabel('Frequency')
plt.title('Distribution of num_polls')
# long tail

In [ ]:
df_combined['num_schools'].hist(bins=100)

# add labels and title
plt.xlabel('num_schools')
plt.ylabel('Frequency')
plt.title('Distribution of num_schools')

# long tail

In [ ]:
df_combined['num_stops'].hist(bins=100)

# add labels and title
plt.xlabel('num_stops')
plt.ylabel('Frequency')
plt.title('Distribution of num_stops')

# long tail

In [ ]:
df_combined['num_earners'].hist(bins=100)

# add labels and title
plt.xlabel('num_earners')
plt.ylabel('Frequency')
plt.title('Distribution of num_earners')
# quite even

In [ ]:
df_combined['median_age'].hist(bins=100)

# add labels and title
plt.xlabel('median_age')
plt.ylabel('Frequency')
plt.title('Distribution of median_age')
# quite even

In [ ]:
df_combined['median_income'].hist(bins=100)

# add labels and title
plt.xlabel('median_income')
plt.ylabel('Frequency')
plt.title('Distribution of median_income')
# long tail on both side

In [ ]:
df_combined['total_businesses'].hist(bins=100)

# add labels and title
plt.xlabel('total_businesses')
plt.ylabel('Frequency')
plt.title('Distribution of total_businesses')
# long tail

In [ ]:
df_combined['total_turnover'].hist(bins=100)

# add labels and title
plt.xlabel('total_turnover')
plt.ylabel('Frequency')
plt.title('Distribution of total_turnover')
# long tail 

In [ ]:
df_combined['num_0_19_people'].hist(bins=100)

# add labels and title
plt.xlabel('num_0_19_people')
plt.ylabel('Frequency')
plt.title('Distribution of num_0_19_people')
# looks good?

In [ ]:
df_combined['young_perc'].hist(bins=100)

# add labels and title
plt.xlabel('young_perc')
plt.ylabel('Frequency')
plt.title('Distribution of young_perc')
# skewed?

In [ ]:
df_combined['total_people'].hist(bins=1000)

# add labels and title
plt.xlabel('total_people')
plt.ylabel('Frequency')
plt.title('Distribution of total_people')
# skewed?

In [ ]:
df_combined[df_combined.num_earners.notnull()].describe()

In [ ]:
df_combined[df_combined.young_perc.notnull()].describe()

In [ ]:
df_combined[df_combined.total_people>=100].describe()

## 2.2 Create Z-scores

In [ ]:
conn.execute(
	'''
	drop table if exists combined_z_score;
    create table combined_z_score as

    with stats as (
    select avg(num_earners/area_size) as earners_avg
    ,stddev(num_earners/area_size) as earners_stdev
    ,avg(median_age) as age_avg
    ,stddev(median_age) as age_stdev
    ,avg(median_income) as income_avg
    ,stddev(median_income) as income_stdev
    ,avg(total_businesses/area_size) as num_business_avg
    ,stddev(total_businesses/area_size) as num_business_stdev
    ,avg(total_turnover/area_size) as turnover_avg
    ,stddev(total_turnover/area_size) as turnover_stdev
    ,avg(num_0_19_people/area_size) as young_num_avg
    ,stddev(num_0_19_people/area_size) as young_num_stdev
    ,avg(Young_perc) as young_perc_avg
    ,stddev(Young_perc) as young_perc_stdev
    ,avg(num_polls/area_size) as polls_avg
    ,stddev(num_polls/area_size) as polls_stdev
    ,avg(case when num_0_19_people = 0 then 0 else num_schools/num_0_19_people*1000 end) as schools_avg
    ,stddev(case when num_0_19_people = 0 then 0 else num_schools/num_0_19_people*1000 end) as schools_stdev
    ,avg(num_stops/area_size) as stops_avg
    ,stddev(num_stops/area_size) as stops_stdev
    from combined
    where total_people >= 100
    )
    select "SA2_CODE21","SA2_NAME21","SA3_NAME21","SA4_NAME21","STE_NAME21"
    , (num_earners/area_size - earners_avg)/earners_stdev as earners_zscore
    , (median_age - age_avg)/age_stdev as age_zscore
    , (median_income - income_avg)/income_stdev as income_zscore
    , (total_businesses/area_size - num_business_avg)/num_business_stdev as business_zscore
    , (total_turnover/area_size - turnover_avg)/turnover_stdev as turnover_zscore
    , (num_0_19_people/area_size - young_num_avg)/young_num_stdev as yound_population_zscore
    , (young_perc - young_perc_avg)/young_perc_stdev as young_perc_zscore
    , (num_polls/area_size - polls_avg)/polls_stdev as polls_zscore
    , (case when num_0_19_people = 0 then 0 else num_schools/num_0_19_people*1000 end - schools_avg)/schools_stdev as schools_zscore
    , (num_stops/area_size - stops_avg)/stops_stdev as stops_zscore
    , geom
    from combined
		join stats s on 1=1
    where total_people >=100
		
    ''')
df_combined_z_score = query(conn, "select * from combined_z_score")
df_combined_z_score.head()

In [30]:
df_combined_z_score = query(conn, "select * from combined_z_score")

In [ ]:
df_combined_z_score.dtypes

In [ ]:
df_combined_z_score.describe()

## 2.3 Sigmoid Function

In [31]:
# import math

# def sigmoid(x):
#   return 1 / (1 + math.exp(-x))
import numpy as np

def sigmoid(x):  
    return np.exp(-np.logaddexp(0, -x))

In [32]:
df_combined_z_score['sigmoid_result'] =sigmoid(df_combined_z_score['earners_zscore'] 
                                    + df_combined_z_score['age_zscore']                 
                                    + df_combined_z_score['income_zscore']              
                                    + df_combined_z_score['business_zscore']            
                                    + df_combined_z_score['turnover_zscore']            
                                    + df_combined_z_score['yound_population_zscore']    
                                    + df_combined_z_score['young_perc_zscore']          
                                    + df_combined_z_score['polls_zscore']               
                                    + df_combined_z_score['schools_zscore']             
                                    + df_combined_z_score['stops_zscore'] )

/Applications/anaconda3/lib/python3.8/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: invalid value encountered in logaddexp
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [33]:
df_combined_z_score[['geom','sigmoid_result']].describe()

,sigmoid_result
count,355.000000
mean,0.455548
std,0.373206
min,0.000026
25%,0.067017
50%,0.413568
75%,0.840829
max,1.000000


In [ ]:
df_combined_z_score.to_sql('task_2_result', conn, if_exists='replace', index=False, dtype={'geom': Geometry('Multipolygon', srid)})
query(conn, "select * from task_2_result")

In [37]:
# save output to csv
df_combined_z_score.to_csv('Task Data Output/task_2_reult.csv')

### Push to SQL

# plot

In [ ]:
task2 = gpd.read_postgis('''select sigmoid_result,geom from task_2_result''', conn, crs=4326)
task2.plot(figsize=(20, 20),cmap='Set2',categorical=True,
                legend=True,)

In [ ]:
# !pip install polars
from shapely.geometry import Point
import polars as pl

In [ ]:
(
    ggplot()
    + geom_map(task2, fill=None)
    # + geom_map(alerts_geo_df, aes(fill="alert_solved"), size=2)
    # + geom_map(stations_geo_df, colour="yellow", size=3)
    + labs(
        title="Alerts solved y/n, by location + Stations (yellow)",
        caption = "Data from  2023-01-01 to 2023-02-01",
    )
)

In [ ]:
# def calc_color(data, color=None):
#         if color   == 1: 
#             color_sq =  ['#dadaebFF','#bcbddcF0','#9e9ac8F0','#807dbaF0','#6a51a3F0','#54278fF0']; 
#             colors = 'Purples';
#         elif color == 2: 
#             color_sq = ['#c7e9b4','#7fcdbb','#41b6c4','#1d91c0','#225ea8','#253494']; 
#             colors = 'YlGnBu';
#         elif color == 3: 
#             color_sq = ['#f7f7f7','#d9d9d9','#bdbdbd','#969696','#636363','#252525']; 
#             colors = 'Greys';
#         elif color == 9: 
#             color_sq = ['#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000'];
                        
#         else:           
#             color_sq = ['#ffffd4','#fee391','#fec44f','#fe9929','#d95f0e','#993404']; 
#             colors = 'YlOrBr';
#         new_data, bins = pd.qcut(data, 6, retbins=True, 
#         labels=list(range(6)))
#         color_ton = []
#         for val in new_data:
#             color_ton.append(color_sq[val]) 
#         if color != 9:
#             colors = sns.color_palette(colors, n_colors=6)
#             sns.palplot(colors, 0.6);
#             for i in range(6):
#                 print ("\n"+str(i+1)+': '+str(int(bins[i]))+
#                        " => "+str(int(bins[i+1])-1))
#             print("\n\n   1   2   3   4   5   6")    
#         return color_ton, bins

In [ ]:
# def plot_cities_data(sf, title, cities, data=None,color=None, print_id=False):
 
#     color_ton, bins = calc_color(data, color)
#     df = read_shapefile(sf)
#     city_id = []
#     for i in cities:
#         city_id.append(df[df.DIST_NAME == 
#                             i.upper()].index.get_values()[0])
#     plot_map_fill_multiples_ids_tone(sf, title, city_id, 
#                                      print_id, 
#                                      color_ton, 
#                                      bins, 
#                                      x_lim = None, 
#                                      y_lim = None, 
#                                      figsize = (11,9));
# def plot_map_fill_multiples_ids_tone(sf, title, city,  
#                                      print_id, color_ton, 
#                                      bins, 
#                                      x_lim = None, 
#                                      y_lim = None, 
#                                      figsize = (11,9)):
   
        
#     plt.figure(figsize = figsize)
#     fig, ax = plt.subplots(figsize = figsize)
#     fig.suptitle(title, fontsize=16)
#     for shape in sf.shapeRecords():
#         x = [i[0] for i in shape.shape.points[:]]
#         y = [i[1] for i in shape.shape.points[:]]
#         ax.plot(x, y, 'k')
            
#     for id in city:
#         shape_ex = sf.shape(id)
#         x_lon = np.zeros((len(shape_ex.points),1))
#         y_lat = np.zeros((len(shape_ex.points),1))
#         for ip in range(len(shape_ex.points)):
#             x_lon[ip] = shape_ex.points[ip][0]
#             y_lat[ip] = shape_ex.points[ip][1]
#         ax.fill(x_lon,y_lat, color_ton[city.index(id)])
#         if print_id != False:
#             x0 = np.mean(x_lon)
#             y0 = np.mean(y_lat)
#             plt.text(x0, y0, id, fontsize=10)
#     if (x_lim != None) & (y_lim != None):     
#         plt.xlim(x_lim)
#         plt.ylim(y_lim)


#  Task 3

## POI

In [ ]:
path_to_file = '/Users/apple/Desktop/Data2901/Assignment/DATA2901--2024S1/Data/Points_Of_Interest_EPSG4326.json'

In [ ]:
import geojson
with open(path_to_file) as f:
    gj = geojson.load(f)
# features = gj['features'][0]

In [122]:
df_poi = pd.json_normalize(gj['Points_Of_Interest']['features'],meta=['type', ['features', 'type'], ['features', 'geometry']])
print(df_poi.shape)
df_poi.head()

(143405, 21)


,type,geometry.type,geometry.coordinates,properties.topoid,properties.poigroup,properties.poitype,properties.poiname,properties.poilabel,properties.poilabeltype,properties.poialtlabel,...,properties.accesscontrol,properties.startdate,properties.enddate,properties.lastupdate,properties.msoid,properties.centroidid,properties.shapeuuid,properties.changetype,properties.processstate,properties.urbanity
0,Feature,Point,"[152.122022, -31.106163]",500000000,9,Mine - Underground,None,Mine - Underground,GENERIC,None,...,1,20210811075603,30000101000000,20210811075657,233046,None,729e2b57-0cd4-3f70-90fa-9dce09e34a8e,I,None,S
1,Feature,Point,"[152.298688, -31.021475]",500005504,3,Lookout,KUNDERANG LOOKOUT,KUNDERANG LOOKOUT,NAMED,None,...,1,20100927115312,30000101000000,20100927115312.535000,83091,None,d88a28a8-c572-3992-995f-d26a274aea18,I,None,S
2,Feature,Point,"[152.337856, -31.015759]",500005505,3,Lookout,FALLS LOOKOUT,FALLS LOOKOUT,NAMED,None,...,1,20100927115312,30000101000000,20100927115312.535000,83691,None,21b476d2-6519-3e28-8b19-1526fcb9652f,I,None,S
3,Feature,Point,"[152.341813, -31.018974]",500005507,3,Lookout,MCCOYS LOOKOUT,MCCOYS LOOKOUT,NAMED,None,...,1,20100927115312,30000101000000,20100927115312.535000,83380,None,016d69b2-6530-39e7-89a6-6e3054df55ac,I,None,S
4,Feature,Point,"[152.478825, -31.20754]",500012781,3,Picnic Area,WILSON RIVER PICNIC AREA,WILSON RIVER PICNIC AREA,NAMED,None,...,1,20201223091118,30000101000000,20201223091146.360000,231054,None,49ad26c8-609e-3aa0-b4ad-51459b43ab51,M,None,S


In [ ]:
df_poi.describe()

,properties.topoid,properties.poigroup,properties.poisourcefeatureoid,properties.accesscontrol,properties.msoid
count,1.434050e+05,143405.000000,143405.000000,143405.0,143405.000000
mean,5.110120e+08,2.972958,58.432851,1.0,142297.717548
std,5.618819e+06,2.325082,49.029415,0.0,70086.125259
min,5.000000e+08,1.000000,1.000000,1.0,1.000000
25%,5.051353e+08,1.000000,13.000000,1.0,83978.000000
50%,5.130093e+08,3.000000,48.000000,1.0,155971.000000
75%,5.156241e+08,4.000000,84.000000,1.0,203160.000000
max,5.184452e+08,9.000000,166.000000,1.0,242107.000000


In [125]:
# replace column name to make it more standardized
df_poi.columns = df_poi.columns.str.replace('.','_')
df_poi.dtypes

type                              object
geometry_type                     object
geometry_coordinates              object
properties_topoid                  int64
properties_poigroup                int64
properties_poitype                object
properties_poiname                object
properties_poilabel               object
properties_poilabeltype           object
properties_poialtlabel            object
properties_poisourcefeatureoid     int64
properties_accesscontrol           int64
properties_startdate              object
properties_enddate                object
properties_lastupdate             object
properties_msoid                   int64
properties_centroidid             object
properties_shapeuuid              object
properties_changetype             object
properties_processstate           object
properties_urbanity               object
dtype: object

In [107]:
df_poi['type'].unique()

array(['Feature'], dtype=object)

In [119]:
df_poi['geometry_type'].unique()

array(['Point'], dtype=object)

In [121]:
len(df_poi['properties_topoid'].unique())
#  unique id

143405

In [132]:
df_poi['geometry_coordinates'].unique
# unique coordinates

<bound method Series.unique of 0         [152.122022, -31.106163]
1         [152.298688, -31.021475]
2         [152.337856, -31.015759]
3         [152.341813, -31.018974]
4          [152.478825, -31.20754]
                    ...           
143400    [151.109504, -33.938152]
143401    [151.110133, -33.938101]
143402    [151.108852, -33.938194]
143403    [151.108207, -33.938241]
143404    [151.100007, -33.939702]
Name: geometry_coordinates, Length: 143405, dtype: object>

In [137]:
# keep only useful columns
df_poi_clean = df_poi[['geometry_coordinates','properties_topoid','properties_poitype','properties_poiname']]

### Load to SQL

In [138]:
conn.execute('''
drop table if exists poi_raw;
create table poi_raw (
geometry_coordinates              varchar(100) not null primary key,
properties_topoid                  int,
properties_poitype                varchar(100),
properties_poiname                varchar(100)
);             
''')
df_poi_clean.to_sql('poi_raw', conn, if_exists='replace', index=False, dtype={'geom': Geometry('POINT', srid)})
query(conn, "select * from poi_raw")

,geometry_coordinates,properties_topoid,properties_poitype,properties_poiname
0,"{152.122022,-31.106163}",500000000,Mine - Underground,None
1,"{152.298688,-31.021475}",500005504,Lookout,KUNDERANG LOOKOUT
2,"{152.337856,-31.015759}",500005505,Lookout,FALLS LOOKOUT
3,"{152.341813,-31.018974}",500005507,Lookout,MCCOYS LOOKOUT
4,"{152.478825,-31.20754}",500012781,Picnic Area,WILSON RIVER PICNIC AREA
...,...,...,...,...
143400,"{151.109504,-33.938152}",518445230,Roadside Emergency Telephone,MEP3631E
143401,"{151.110133,-33.938101}",518445231,Roadside Emergency Telephone,MEP3621E
143402,"{151.108852,-33.938194}",518445232,Roadside Emergency Telephone,MEP3632E
143403,"{151.108207,-33.938241}",518445233,Roadside Emergency Telephone,MEP3641E


In [139]:
conn.execute('''
             drop table if exists poi;
create table poi as
             select 
cast(replace(replace(replace(geometry_coordinates,'{','Point('),'}',')'),',',' ') as GEOMETRY(POINT,4326)) geometry_coordinates,
properties_topoid,
properties_poitype,
properties_poiname
             from poi_raw
''')
query(conn, "select * from poi")

,geometry_coordinates,properties_topoid,properties_poitype,properties_poiname
0,0101000020E61000008F6CAE9AE7036340950B957F2D1B...,500000000,Mine - Underground,None
1,0101000020E6100000A5F622DA8E0963407DAEB6627F05...,500005504,Lookout,KUNDERANG LOOKOUT
2,0101000020E61000003CD862B7CF0A63401F9E25C80804...,500005505,Lookout,FALLS LOOKOUT
3,0101000020E6100000200BD121F00A63406C79E57ADB04...,500005507,Lookout,MCCOYS LOOKOUT
4,0101000020E61000003B70CE88520F6340A29C68572135...,500012781,Picnic Area,WILSON RIVER PICNIC AREA
...,...,...,...,...
143400,0101000020E6100000FF58880E81E36240A8565F5D15F8...,518445230,Roadside Emergency Telephone,MEP3631E
143401,0101000020E6100000BB26A43586E3624026AC8DB113F8...,518445231,Roadside Emergency Telephone,MEP3621E
143402,0101000020E6100000568330B77BE36240D7A6B1BD16F8...,518445232,Roadside Emergency Telephone,MEP3632E
143403,0101000020E610000058C6866E76E362401762F54718F8...,518445233,Roadside Emergency Telephone,MEP3641E


## 2. Green

In [79]:
file_path = 'Task 3 Data/green_cleaned/green.shp'
df_green = gpd.read_file(file_path)
print(df_green.shape)
df_green.head()

(12978, 4)


,ASSET_NAME,LAY_CLASS,weights,geometry
0,Crown Reserve 55 (Part),Undeveloped or Unspecified,0.00,"POLYGON ((150.49420 -33.72286, 150.49414 -33.7..."
1,Albert Sloss Reserve,Civic,1.00,"POLYGON ((151.21631 -33.87802, 151.21624 -33.8..."
2,Mount Kuring-Gai Park,Bushland,-0.25,"POLYGON ((151.13290 -33.66195, 151.13268 -33.6..."
3,Jardine Park,Sports,1.00,"POLYGON ((150.90518 -33.94474, 150.90528 -33.9..."
4,Bray Street Reserve,Parks and Gardens,0.75,"POLYGON ((151.05453 -33.80032, 151.05433 -33.8..."


In [80]:
df_green.describe()

,weights
count,12978.000000
mean,0.534096
std,0.446590
min,-0.250000
25%,0.000000
50%,0.750000
75%,0.750000
max,1.000000


In [81]:
df_green.dtypes

ASSET_NAME      object
LAY_CLASS       object
weights        float64
geometry      geometry
dtype: object

In [83]:
df_green.geometry.unique

<bound method Series.unique of 0        POLYGON ((150.49420 -33.72286, 150.49414 -33.7...
1        POLYGON ((151.21631 -33.87802, 151.21624 -33.8...
2        POLYGON ((151.13290 -33.66195, 151.13268 -33.6...
3        POLYGON ((150.90518 -33.94474, 150.90528 -33.9...
4        POLYGON ((151.05453 -33.80032, 151.05433 -33.8...
                               ...                        
12973    POLYGON ((151.16093 -33.72650, 151.16081 -33.7...
12974    POLYGON ((150.38961 -33.50124, 150.38889 -33.5...
12975    POLYGON ((151.22260 -33.84441, 151.22261 -33.8...
12976    POLYGON ((151.05595 -33.95093, 151.05587 -33.9...
12977    POLYGON ((150.77285 -33.76626, 150.77291 -33.7...
Name: geometry, Length: 12978, dtype: geometry>

In [87]:
conn.execute('''
drop table if exists task_3_green;
create table task_3_green (
asset_name              varchar(100) not null,
lay_class                  varchar(100),
weights             float,
geom                GEOMETRY(MULTIPOLYGON,4326) primary key
);             
''')
df_green.to_sql('schools', conn, if_exists='replace', index=False, dtype={'geom': Geometry('MULTIPOLYGON', srid)})
query(conn, "select * from task_3_green")

ProgrammingError: (psycopg2.ProgrammingError) can't adapt type 'Polygon'
[SQL: INSERT INTO schools ("ASSET_NAME", "LAY_CLASS", weights, geometry) VALUES (%(ASSET_NAME)s, %(LAY_CLASS)s, %(weights)s, %(geometry)s)]
[parameters: ({'ASSET_NAME': 'Crown Reserve 55 (Part)', 'LAY_CLASS': 'Undeveloped or Unspecified', 'weights': 0.0, 'geometry': <POLYGON ((150.494 -33.723, 150.494 -33.723, 150.494 -33.723, 150.494 -33.72...>}, {'ASSET_NAME': 'Albert Sloss Reserve', 'LAY_CLASS': 'Civic', 'weights': 1.0, 'geometry': <POLYGON ((151.216 -33.878, 151.216 -33.878, 151.216 -33.878, 151.216 -33.87...>}, {'ASSET_NAME': 'Mount Kuring-Gai Park', 'LAY_CLASS': 'Bushland', 'weights': -0.25, 'geometry': <POLYGON ((151.133 -33.662, 151.133 -33.662, 151.133 -33.662, 151.133 -33.66...>}, {'ASSET_NAME': 'Jardine Park', 'LAY_CLASS': 'Sports', 'weights': 1.0, 'geometry': <POLYGON ((150.905 -33.945, 150.905 -33.945, 150.906 -33.945, 150.906 -33.94...>}, {'ASSET_NAME': 'Bray Street Reserve', 'LAY_CLASS': 'Parks and 

## 3. School Data

### School Locations and Ranking

First step is to load in all the required Packages

#### Location Dataset

Next we need the list of schools (names) with locations attached. This includes all 3300 schools in NSW (Make sure the related CSV is in the same directory as this notebook). This is read in, its two coordinates columns converted into a single geometry column, and finally stripped back to just contain the necessary data.

In [8]:
Initial_data_path = 'Task 3 Data/Scraped_School_data/NSW_Schools.csv'
school_location_data = pd.read_csv(Initial_data_path)
latitudes = school_location_data['Latitude']
longitudes = school_location_data['Longitude']
school_location_data = gpd.GeoDataFrame(school_location_data, geometry=gpd.points_from_xy(longitudes, latitudes)) #Pairs together Longitude Latitude pairs and turns them into geometry points
school_location_data = school_location_data[['School Name','geometry']] # Keep only important columns
school_location_data.head()

,School Name,geometry
0,A.G.B.U. Alexander Primary School,POINT (151.18715 -33.67679)
1,Abbotsford Public School,POINT (151.13100 -33.85270)
2,Abbotsleigh,POINT (151.11100 -33.71990)
3,Abbotsleigh Junior School,POINT (151.11300 -33.71620)
4,Aberdeen Public School,POINT (150.88800 -32.16610)


### Scraping the Ranking Data (School Names)

Next we begin scraping the data containing the HSC 'results' of numerous schools in NSW.

In [9]:
basic_URL = "https://bettereducation.com.au/Results/Hsc.aspx" #This is where the data is coming from
try:
    webpage_source = requests.get(basic_URL).text
except:
    print('Error Occured: Request Failed')
try:
    content = BeautifulSoup(webpage_source, 'html5lib')
except:
        print('Error Occured: Parsing Failed')

This webpage contains a table with ~200 schools, where the data is contained within tags of a similar id. To efficiently extract these I created a function that takes in a number (Row Number) and returns the related id for that school's name.

In [10]:
def id_constructor(start='ctl00_ContentPlaceHolder1_GridView1_ctl',end='_HyperLinkSchool',number='0'):
    new_id = str(start) + str(number) + str(end)
    return new_id

Next we use this function to get the id's for all the rows of the table. This is where we go overboard, and create id's for up to 300 schools, as the exact number of rows in the table is somewhat confusing actually know. On the website you can view 299 schools, but the table says it only has 205.

In [11]:
ids = [] #This will be used each time we iterate over the table.
for i in range(300):
    ids.append(str(str(i).zfill(2))) # zfill is used to pad with 0s required by the tag.

name_ids = []
for i in ids:
    name_ids.append(id_constructor(number=i))

Now we come to actually extracting the names, and storing them in a list that will later be turned into a database column.

In [12]:
school_names_rankings = []
for code in name_ids:
    try:    # try loop is used in case we have got too many ids, and we try to access a row that doesn't exist.
        name = content.find('a', id=code).text
        school_names_rankings.append(name)
    except:
        continue

By checking the length I can check that I have extracted 205 school names, which is partially what is expected from this table. So long as this remains consistent, everything is fine.

In [13]:
len(school_names_rankings)

205

### Scraping the Ranking Data (%DAs)

Next step is to repeat the above process, only this time extracting the data on 'Percentage of exams sat that achieved a DA'. This will be our metric of school success. This begins by creating all the ids for each row of this column of data.

In [14]:
percentDA_ids = []
for i in ids:
    percentDA_ids.append(id_constructor(start='ctl00_ContentPlaceHolder1_GridView1_ctl', end='_lblPercent', number=i))

As before I have a list to store all the results that will later be turned into a column. Iterate over the each of the ids from before to get each entry.

In [15]:
school_percentageDA = []
for code in percentDA_ids:
    try:    # Once again using a try to account for the excess of ids.
        percent_DA = content.find('span', id=code).text
        school_percentageDA.append(percent_DA)
    except:
        continue

### Scraping the HSC Students Data

In addition to the %DA, the website contains information on the number of enrolled HSC students of each school mentioned. This is valuable in determining how 'Bustling' a school is, as more students strongly implies the school is more bustling.

In [16]:
hscStudents_ids = []
for i in ids:
    hscStudents_ids.append(id_constructor(end='_lblStudents', number=i))

Store all the Student Numbers in a list that will be added as a column, iterating over each of the ids made above.

In [17]:
school_HSCstudents = []
for code in hscStudents_ids:
    try:    # Once again using a try to account for the excess of ids.
        HSCstudents = content.find('span', id=code).text
        school_HSCstudents.append(HSCstudents)
    except:
        continue

### Scraping the number of Exams Sat

While the number of students is a good indicator of bustling, 'bustling' is better defined as full of activity, and what activity better than sitting tests. So this will be another good indicator of how 'bustling' the schools actually are.

In [18]:
testSat_ids = []
for i in ids:
    testSat_ids.append(id_constructor(end='_lblExamsSat', number=i))

Once again using the ids to extract each table entry.

In [19]:
school_testSat = []
for code in testSat_ids:
    try:    # Once again using a try to account for the excess of ids.
        testSat = content.find('span', id=code).text
        school_testSat.append(testSat)
    except:
        continue

### Matching Names

Testing for names that are duplicates by turning the list into a set (which removes duplicates). As the lengths are the same, we can assume there are no duplicates.

In [20]:
len(school_names_rankings), len(set(school_names_rankings))

(205, 205)

Attempting to see how many of the school names line up across datasets. Without any alterations, there are 154 that match, and 51 schools that don't match up. These will be the ones we have to identify manually.

In [21]:
matched_schools = list(set(school_names_rankings) & set(school_location_data['School Name']))
unmatched_schools = list(set(school_names_rankings) - set(school_location_data['School Name']))
len(matched_schools), len(unmatched_schools)

(154, 51)

In [25]:
unmatched_schools_df = pd.DataFrame(unmatched_schools)
unmatched_schools_df.to_csv('Task Data Output/Task 3 - Schools_to_be_renamed.csv')

In [29]:
unmatched_schools_correct_names = pd.read_csv('Task Data Output/Task 3 - Schools_to_be_renamed.csv')
unmatched_schools_correct_names = unmatched_schools_correct_names.drop(['Unnamed: 0'], axis=1) # Remove the extra 
unmatched_schools_correct_names

,0
0,"St John XXIII Catholic College,Stanhope Garden..."
1,"St Augustine's College,Brookvale,NSW,2100"
2,Oakhill College Castle Hill
3,Penrith Selective High School
4,St Patrick's Marist College (Dundas)
5,Canberra Grammar School
6,Smith's Hill High School
7,St Mary's Cathedral College (Sydney)
8,Broughton Anglican College (Menangle Park)
9,Parramatta Marist High School (Westmead)


A new file containing the incorrect names paired with the correct names was painfully put together, so as to ensure correct mapping, as any level of automation ended up resulting in some incorrections.

In [30]:
unmatched_schools_correct_names = pd.read_csv('Task 3 Data/Schools_with_new_names.csv')
unmatched_schools_correct_names = unmatched_schools_correct_names.drop(['Unnamed: 0'], axis=1) # Remove the extra 
incorrect_names = list(unmatched_schools_correct_names['Original Name'])
correct_names = list(unmatched_schools_correct_names['New Name'])

Go through each incorrect name in the original list and replace it with the correct one.

In [31]:
for i in range(len(incorrect_names)):
    index = school_names_rankings.index(incorrect_names[i])
    school_names_rankings[index] = correct_names[i]

We expect to have 203 schools as 2 schools could not be matched (Actual location ACT, not NSW).

In [32]:
len(set(school_names_rankings) & set(school_location_data['School Name']))

203

### Storing Data

Storing the data in a dataframe, which will soon be attached to the location GeoDataFrame.

In [33]:
school_ranking_data = {
    'school_name': school_names_rankings,
    'HSC_students': school_HSCstudents,
    'exams_sat': school_testSat,
    'school_achievement': school_percentageDA
}
school_ranking_data = pd.DataFrame(school_ranking_data)
school_ranking_data.head()

,school_name,HSC_students,exams_sat,school_achievement
0,North Sydney Boys High School,158,955,69.0%
1,James Ruse Agricultural High School,158,996,60.0%
2,Baulkham Hills High School,200,1210,55.8%
3,North Sydney Girls High School,153,909,54.7%
4,Reddam House,122,684,54.2%


### Combining Data

The next step is to actually combine the two datasets, we've been thorough in the previous step to ensure that they can be properly joined by the name columns.

In [70]:
school_complete_data = school_ranking_data.merge(
    right = school_location_data, 
    how = 'inner',
    left_on = 'school_name',
    right_on = 'School Name')
school_complete_data = school_complete_data.drop(['School Name'], axis=1) # Remove the duplicate column
school_complete_data = gpd.GeoDataFrame(school_complete_data) # Make the dataframe a Geodataframe
school_complete_data.set_geometry('geometry',inplace=True) # Make sure the correct column is the geometry column
school_complete_data.head()

,school_name,HSC_students,exams_sat,school_achievement,geometry
0,North Sydney Boys High School,158,955,69.0%,POINT (151.20700 -33.82960)
1,James Ruse Agricultural High School,158,996,60.0%,POINT (151.04300 -33.78060)
2,Baulkham Hills High School,200,1210,55.8%,POINT (150.99100 -33.75020)
3,North Sydney Girls High School,153,909,54.7%,POINT (151.20300 -33.83020)
4,Reddam House,122,684,54.2%,POINT (151.25300 -33.88930)


Finally with our complete data we can save it somewhere.

In [71]:
school_complete_data.to_file('Task Data Output/Task 3 - School_complete_data.shp')

<ipython-input-71-5911dbf290ec>:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  school_complete_data.to_file('Task Data Output/Task 3 - School_complete_data.shp')


To test whether it was correctly saved, we can load in the file and check the details.

In [72]:
school_complete_data2 = gpd.read_file('Task Data Output/Task 3 - School_complete_data.shp')
school_complete_data2

,school_nam,HSC_studen,exams_sat,school_ach,geometry
0,North Sydney Boys High School,158,955,69.0%,POINT (151.20700 -33.82960)
1,James Ruse Agricultural High School,158,996,60.0%,POINT (151.04300 -33.78060)
2,Baulkham Hills High School,200,1210,55.8%,POINT (150.99100 -33.75020)
3,North Sydney Girls High School,153,909,54.7%,POINT (151.20300 -33.83020)
4,Reddam House,122,684,54.2%,POINT (151.25300 -33.88930)
...,...,...,...,...,...
200,Broughton Anglican College,96,531,5.8%,POINT (150.76800 -34.09970)
201,St Columba's Catholic College,120,659,5.8%,POINT (150.58800 -33.67320)
202,Cedars Christian College,45,228,5.7%,POINT (150.82900 -34.45550)
203,St Philip's Christian College - Port Stephens,44,232,5.6%,POINT (152.11000 -32.73830)


In [65]:
school_complete_data2.dtypes

school_nam      object
HSC_studen      object
exams_sat       object
school_ach      object
geometry      geometry
dtype: object

### To SQL

In [75]:
conn.execute('''
drop table if exists task_3_school_data;
create table task_3_school_data (
             school_name varchar(100),
             hsc_student int,
             exam_sat int,
             school_achievement float,
             geom GEOMETRY(POINT,4326)
);
''')
# school_complete_data2.to_sql('task_3_school_data', conn, if_exists='replace', index=False, dtype={'geom': Geometry('POINT', srid)})
# query(conn, "select * from task_3_school_data")

In [76]:
school_complete_data2.to_sql('task_3_school_data', conn, if_exists='replace', index=False, dtype={'geom': Geometry('POINT', srid)})
# query(conn, "select * from task_3_school_data")

ProgrammingError: (psycopg2.ProgrammingError) can't adapt type 'Point'
[SQL: INSERT INTO task_3_school_data (school_nam, "HSC_studen", exams_sat, school_ach, geometry) VALUES (%(school_nam)s, %(HSC_studen)s, %(exams_sat)s, %(school_ach)s, %(geometry)s)]
[parameters: ({'school_nam': 'North Sydney Boys High School', 'HSC_studen': '158', 'exams_sat': '955', 'school_ach': '69.0%', 'geometry': <POINT (151.207 -33.83)>}, {'school_nam': 'James Ruse Agricultural High School', 'HSC_studen': '158', 'exams_sat': '996', 'school_ach': '60.0%', 'geometry': <POINT (151.043 -33.781)>}, {'school_nam': 'Baulkham Hills High School', 'HSC_studen': '200', 'exams_sat': '1210', 'school_ach': '55.8%', 'geometry': <POINT (150.991 -33.75)>}, {'school_nam': 'North Sydney Girls High School', 'HSC_studen': '153', 'exams_sat': '909', 'school_ach': '54.7%', 'geometry': <POINT (151.203 -33.83)>}, {'school_nam': 'Reddam House', 'HSC_studen': '122', 'exams_sat': '684', 'school_ach': '54.2%', 'geometry': <POINT (151.253